# SuspiciousLogin Dataset — Dataset Statistics (Article 1)

Coverage, volume, class distribution, per-user activity concentration (Gini index, Lorenz curve), and data quality statistics for the published dataset (`suspicious_logins_public_v1.csv`). Produces the numbers and figures for the dataset paper's descriptive section.

Run against the **public** schema -- this notebook does not need `event_time` or any restricted-only column.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PUBLIC_FILE = Path("data/processed/suspicious_logins_public_v1.csv")
if not PUBLIC_FILE.exists():
    PUBLIC_FILE = Path("suspicious_logins_public_v1.csv")

df = pd.read_csv(PUBLIC_FILE)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns from {PUBLIC_FILE}")

## 2. Coverage and volume

In [ ]:
n_events = len(df)
n_users = df["actor_pseudo_id"].nunique()
n_countries = df["ip_country"].nunique()
n_continents = df["continent"].nunique()

print(f"Total events: {n_events:,}")
print(f"Distinct pseudonymized users: {n_users:,}")
print(f"Distinct countries (public schema, rare ones generalized to 'Other'): {n_countries}")
print(f"Distinct continents: {n_continents}")
print(f"Login types: {dict(df['login_type'].value_counts())}")

In [ ]:
# Per-day volume, reconstructed from month/day_of_week/quarter (event_time
# itself is restricted-only) -- an approximate daily count via (month,
# day_of_week) is not exact per calendar day, so this section reports
# monthly volume instead, which the public schema supports directly and
# exactly.
by_month = df.groupby("month").size()
print("Events per month:")
print(by_month)
print(f"\nMean events/month: {by_month.mean():.1f}, median: {by_month.median():.1f}, "
     f"min: {by_month.min()}, max: {by_month.max()}")

## 3. Label distribution, with a 95% confidence interval

In [ ]:
n_positive = df["label"].sum()
positive_rate = n_positive / n_events

# Wilson score interval -- more reliable than the naive normal
# approximation for a proportion this far from 0.5.
z = 1.96
denom = 1 + z**2 / n_events
center = (positive_rate + z**2 / (2 * n_events)) / denom
margin = z * np.sqrt(positive_rate * (1 - positive_rate) / n_events + z**2 / (4 * n_events**2)) / denom
ci_low, ci_high = center - margin, center + margin

print(f"Positive (suspicious) events: {n_positive:,} of {n_events:,} ({positive_rate*100:.2f}%)")
print(f"95% Wilson confidence interval: [{ci_low*100:.2f}%, {ci_high*100:.2f}%]")

## 4. Per-user activity distribution, concentration, and the Gini index

In [ ]:
events_per_user = df["actor_pseudo_id"].value_counts()

print("Events per user -- distribution:")
print(events_per_user.describe())
print()
for p in [50, 75, 90, 95, 99]:
    print(f"  p{p}: {events_per_user.quantile(p/100):.0f}")
print(f"  max: {events_per_user.max()}")

In [ ]:
def gini_index(values):
    """Standard Gini coefficient via the sorted-cumulative-share
    formula. 0 = perfectly equal (everyone has the same number of
    events), 1 = maximally unequal (one user has everything)."""
    sorted_values = np.sort(values)
    n = len(sorted_values)
    cumulative = np.cumsum(sorted_values)
    return (2 * np.sum((np.arange(1, n + 1)) * sorted_values) - (n + 1) * cumulative[-1]) / (n * cumulative[-1])

gini = gini_index(events_per_user.values)
print(f"Gini index of events-per-user: {gini:.4f}")
print()

total_events = events_per_user.sum()
for pct in [0.01, 0.05, 0.10]:
    top_n = max(1, int(len(events_per_user) * pct))
    share = events_per_user.iloc[:top_n].sum() / total_events
    print(f"Top {pct*100:.0f}% of users ({top_n} user(s)) account for "
         f"{share*100:.1f}% of all events")

single_event_users = (events_per_user == 1).sum()
print(f"\nUsers with exactly 1 event: {single_event_users} "
     f"({single_event_users/len(events_per_user)*100:.1f}% of all users)")

### Figure: distribution of events per user (log scale)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(events_per_user.values, bins=50)
ax.set_yscale("log")
ax.set_xlabel("Events per user")
ax.set_ylabel("Number of users (log scale)")
ax.set_title("Distribution of events per user")
plt.tight_layout()
plt.savefig("events_per_user_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

### Figure: Lorenz curve

In [ ]:
sorted_events = np.sort(events_per_user.values)
cumulative_users = np.arange(1, len(sorted_events) + 1) / len(sorted_events)
cumulative_events = np.cumsum(sorted_events) / sorted_events.sum()

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(cumulative_users, cumulative_events, label="Lorenz curve")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect equality")
ax.set_xlabel("Cumulative share of users")
ax.set_ylabel("Cumulative share of events")
ax.set_title(f"Lorenz curve of user activity (Gini = {gini:.3f})")
ax.legend()
plt.tight_layout()
plt.savefig("lorenz_curve.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. Data quality

In [ ]:
print("Missing values per column (public schema):")
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) == 0:
    print("  none")
else:
    for col, n_missing in missing.items():
        print(f"  {col}: {n_missing} ({n_missing/len(df)*100:.2f}%)")

In [ ]:
# Exact-duplicate check on the public schema itself (deduplication
# already happens upstream in processed.py against the raw files -- this
# re-confirms none survived into the published file).
duplicate_rows = df.duplicated().sum()
print(f"Fully duplicate rows in the public file: {duplicate_rows}")

invalid_label = (~df["label"].isin([0, 1])).sum()
print(f"Rows with an invalid label (not 0 or 1): {invalid_label}")

### Figure: missing-value pattern

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(4, len(missing) * 0.4 + 1)))
if len(missing) > 0:
    ax.barh(missing.index[::-1], (missing.values[::-1] / len(df) * 100))
    ax.set_xlabel("% missing")
    ax.set_title("Missing values by column (only columns with at least one gap)")
else:
    ax.text(0.5, 0.5, "No missing values in any column", ha="center", va="center")
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.savefig("missing_values.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. Geographic and authentication distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_countries = df["ip_country"].value_counts().head(10)
axes[0].barh(top_countries.index[::-1], top_countries.values[::-1])
axes[0].set_xlabel("Events")
axes[0].set_title("Top 10 countries by event count")

login_types = df["login_type"].value_counts()
axes[1].bar(login_types.index, login_types.values)
axes[1].set_ylabel("Events")
axes[1].set_title("Login type distribution")
axes[1].tick_params(axis="x", rotation=30)
for label in axes[1].get_xticklabels():
    label.set_ha("right")

plt.tight_layout()
plt.savefig("geography_and_auth.png", dpi=120, bbox_inches="tight")
plt.show()

**Note on `login_type`**: at the full two-institution scale, three values
appear that were not present (or not yet documented) in earlier,
smaller samples -- `federated_login`, `session_refresh`, and
`admin_login`, alongside the three originally documented
(`google_password`, `reauth`, `exchange`). `docs/DATA_DICTIONARY_EN.md`
should be updated to list all six.

**Note on the single-row data quality edge case**: exactly one row
(0.0014% of the dataset) has `event_time`-derived fields
(`event_hour`, `month`, etc.) and geo fields all missing, most likely
from a missing `event_time` at the source for that one event -- yet a
few derived geographic-change flags (`is_new_country`,
`country_changed`, `geo_jump`) are still populated as if a change was
detected, which is not meaningful when the underlying country itself
is unknown (a pandas NaN-comparison quirk, not a genuine detected
change). Negligible practical impact at this scale, documented here
rather than silently left unmentioned.

## 7. Summary table

In [ ]:
summary = pd.DataFrame([
    {"metric": "Total events", "value": n_events},
    {"metric": "Distinct users", "value": n_users},
    {"metric": "Distinct countries (public)", "value": n_countries},
    {"metric": "Positive rate (%)", "value": round(positive_rate * 100, 2)},
    {"metric": "Positive rate 95% CI low (%)", "value": round(ci_low * 100, 2)},
    {"metric": "Positive rate 95% CI high (%)", "value": round(ci_high * 100, 2)},
    {"metric": "Gini index (events per user)", "value": round(gini, 4)},
    {"metric": "Top 1% of users' share of events (%)", "value": round(
        events_per_user.iloc[:max(1, len(events_per_user)//100)].sum() / total_events * 100, 1)},
    {"metric": "Users with exactly 1 event (%)", "value": round(
        single_event_users / len(events_per_user) * 100, 1)},
    {"metric": "Missing values, any column", "value": int(missing.sum()) if len(missing) else 0},
    {"metric": "Fully duplicate rows", "value": int(duplicate_rows)},
])
summary.to_csv("article1_summary_statistics.csv", index=False)
summary